### Import functions

In [ ]:
from lymphly import *

### Load cohort data

In [ ]:
# Read the table with the classifier's features

feature_table = pd.read_table('Lymphly_feature_table.tsv')

In [ ]:
cohort = 'TEST'
cohorts_settings = {
    'path_to_maf': 'test_data/mutations.maf',
    'path_to_cna_gene': 'test_data/cna-gene.tsv',
    'path_to_annotation': 'test_data/annotation.tsv',
    'name_to_save': 'test_data/Lymphly_test.tsv',
    'ref': 'HG38'
}

In [ ]:
ann = pd.read_csv(cohorts_settings['path_to_annotation'], sep = '\t')
ann = ann.set_index('Sample')

In [ ]:
# Load MAF

maf_load = pd.read_csv(cohorts_settings['path_to_maf'], sep='\t', low_memory=False)
maf_load = format_maf(maf_load, variant_classifications = variant_classification_groups)
if cohorts_settings['ref'] == 'HG19':
    print('Converting_HG19_to_HG38')
    maf_load = convert_hg19_to_hg38(maf_load)

In [ ]:
# Load CNA

if 'path_to_cna_gene' in cohorts_settings.keys():
    cna_gene = pd.read_csv(cohorts_settings['path_to_cna_gene'], sep='\t', index_col=0, low_memory=False)
    cna_gene = check_cna_gene(cna_gene, ann, feature_table)
else:
    cna_gene = None

### Perform classification

In [ ]:
# Set arguments

USE_TRANSLOCATIONS = True
USE_CNA = True
USE_STATUSES = True

In [ ]:
# Cohort classification

lymphly_table = lymphly_classify(
                                  maf=maf_load,
                                  cna_gene=cna_gene,
                                  annotation=ann,
                                  feature_table=feature_table,
                                  use_translocations=USE_TRANSLOCATIONS,
                                  use_cna=USE_CNA,
                                  use_statuses=USE_STATUSES,
                                  **config
                                )

In [ ]:
lymphly_table.to_csv(cohorts_settings['name_to_save'], sep='\t')

### Statistics

In [ ]:
TOP=15
subtype_statistics(lymphly_table, feature_table, top = TOP)

### Subtypes pieplot

In [ ]:
palette_all = pallete_preparation(lymphly_table, palette_pure)
subtypes_pieplot(lymphly_table, palette_all)